In [1]:
# imports
import os
import sys

sys.path.append(os.path.abspath(".."))  # project root

import unittest
from models.bot.AllInBot import AllInBot
from models.bot.BaseBot import BaseBot
from models.GameElements.Game import Game

In [2]:
# static functions
def make_state(bets, my_id=0, stack=1000, folded=()):
    """Builds a minimal server state. bets[i] is player i's current bet; my_id's stack is `stack`."""
    return {
        "players": [
            {
                "id": i,
                "bet": bet,
                "stack": stack if i == my_id else 1000,
                "folded": i in folded,
                "hole_cards": [{"rank": 14, "suit": "hearts"}, {"rank": 13, "suit": "spades"}],
            }
            for i, bet in enumerate(bets)
        ]
    }

In [3]:
# Core Function Testing
class AllInBot_Core_UnitTests(unittest.TestCase):
    """
    Same layout as BotUnitTests in BotUnitTest.ipynb: setUp builds the bot, and only methods starting with 'test_' run.
    AllInBot.choose_action returns ("raise", current_bet + chips) when that beats the highest bet, else ("call", 0).
    """

    def setUp(self):
        self.bot = AllInBot()
        self.bot.gain_chips(1000)

    def test_AllInBot_IsABaseBot(self):
        self.assertIsInstance(self.bot, BaseBot)

    # TESTING choose_action
    def test_NoBetsYet_RaisesAllChips(self):
        state = make_state([0, 0, 0])
        self.assertEqual(self.bot.choose_action(state), ("raise", 1000))

    def test_FacingSmallerBet_RaisesAllChips(self):
        state = make_state([0, 50, 0])
        self.assertEqual(self.bot.choose_action(state), ("raise", 1000))

    def test_AlreadyBetSome_RaiseTargetIsBetPlusChips(self):
        self.bot.place_bet(50)
        self.bot.lose_chips(50)  # 950 behind, 50 in front
        state = make_state([50, 100, 0])
        self.assertEqual(self.bot.choose_action(state), ("raise", 1000))

    def test_StackCannotCoverBet_Calls(self):
        # bot has 1000 total, opponent bet 1500: raising is impossible, so call (all-in call)
        state = make_state([0, 1500, 0])
        self.assertEqual(self.bot.choose_action(state), ("call", 0))

    def test_StackExactlyMatchesBet_Calls(self):
        # all-in target equals the highest bet, so it is not a raise
        state = make_state([0, 1000])
        self.assertEqual(self.bot.choose_action(state), ("call", 0))

    def test_AlreadyAllIn_NoChipsLeft_Calls(self):
        self.bot.place_bet(1000)
        self.bot.lose_chips(1000)  # 0 chips behind, 1000 in front
        state = make_state([1000, 1000])
        self.assertEqual(self.bot.choose_action(state), ("call", 0))

    def test_ZeroChipsAndNoBet_Calls(self):
        self.bot.lose_chips(1000)
        state = make_state([0, 0])
        self.assertEqual(self.bot.choose_action(state), ("call", 0))

    def test_ChooseAction_DoesNotChangeChips(self):
        self.bot.choose_action(make_state([0, 50]))
        self.assertEqual(self.bot.get_chips(), 1000)
        self.assertEqual(self.bot.get_current_bet(), 0)

    # TESTING WITH update_from_state (the server is the source of truth)
    def test_UpdateFromState_ThenChooseAction_UsesServerStack(self):
        state = make_state([0, 20], my_id=0, stack=400)
        self.bot.update_from_state(state, 0)
        self.assertEqual(self.bot.choose_action(state), ("raise", 400))

    def test_UpdateFromState_StackBelowBet_Calls(self):
        state = make_state([0, 500], my_id=0, stack=300)
        self.bot.update_from_state(state, 0)
        self.assertEqual(self.bot.choose_action(state), ("call", 0))

    # TESTING THE ALL-IN AMOUNT WITH BaseBot.all_in
    def test_RaiseTarget_MatchesChipsAllInWouldPutIn(self):
        state = make_state([0, 50])
        action, target = self.bot.choose_action(state)
        put_in = self.bot.all_in(50)
        self.assertEqual(action, "raise")
        self.assertEqual(target, put_in)
        self.assertEqual(self.bot.get_chips(), 0)


unittest.main(argv=[""], exit=False, verbosity=2)

test_AllInBot_IsABaseBot (__main__.AllInBot_Core_UnitTests.test_AllInBot_IsABaseBot) ... ok
test_AlreadyAllIn_NoChipsLeft_Calls (__main__.AllInBot_Core_UnitTests.test_AlreadyAllIn_NoChipsLeft_Calls) ... ok
test_AlreadyBetSome_RaiseTargetIsBetPlusChips (__main__.AllInBot_Core_UnitTests.test_AlreadyBetSome_RaiseTargetIsBetPlusChips) ... ok
test_ChooseAction_DoesNotChangeChips (__main__.AllInBot_Core_UnitTests.test_ChooseAction_DoesNotChangeChips) ... ok
test_FacingSmallerBet_RaisesAllChips (__main__.AllInBot_Core_UnitTests.test_FacingSmallerBet_RaisesAllChips) ... ok
test_NoBetsYet_RaisesAllChips (__main__.AllInBot_Core_UnitTests.test_NoBetsYet_RaisesAllChips) ... ok
test_RaiseTarget_MatchesChipsAllInWouldPutIn (__main__.AllInBot_Core_UnitTests.test_RaiseTarget_MatchesChipsAllInWouldPutIn) ... ok
test_StackCannotCoverBet_Calls (__main__.AllInBot_Core_UnitTests.test_StackCannotCoverBet_Calls) ... ok
test_StackExactlyMatchesBet_Calls (__main__.AllInBot_Core_UnitTests.test_StackExactlyMatch

In [4]:
# Game Function Testing
class AllInBot_Game_UnitTests(unittest.TestCase):
    """
    AllInBots seated in a real Game. Game.betting_round() is not implemented yet, so these tests apply each
    bot's choose_action result by hand (raise -> all_in, call -> call) the way betting_round eventually will.
    """

    def setUp(self):
        self.player_a = AllInBot()
        self.player_b = AllInBot()
        self.player_c = AllInBot()
        self.players = [self.player_a, self.player_b, self.player_c]
        for player in self.players:
            player.gain_chips(1000)

        self.game = Game(self.players)

    def state_from_game(self):
        """Server-style snapshot built from the bots' current chips and bets."""
        return {
            "players": [
                {"id": i, "bet": p.get_current_bet(), "stack": p.get_chips(), "folded": p.is_out, "hole_cards": []}
                for i, p in enumerate(self.players)
            ]
        }

    def play_action(self, bot):
        """Applies one bot's decision to the game: returns the action taken."""
        action, amount = bot.choose_action(self.state_from_game())
        if action == "raise":
            put_in = amount - bot.get_current_bet()
            bot.lose_chips(put_in)
            bot.place_bet(amount)
            self.game.pot += put_in
            self.game.set_table_bet(amount)
        else:
            self.game.pot += bot.call(self.game.table_bet)
        return action

    # TESTING THE BOTS INSIDE A GAME
    def test_AllInBots_AreSeatedInGame(self):
        self.assertEqual(len(self.game.players), 3)
        for player in self.game.players:
            self.assertIsInstance(player, AllInBot)

    def test_StartHand_EachAllInBotGetsTwoCards(self):
        self.game.start_hand()
        for player in self.players:
            self.assertEqual(len(player.get_hand()), 2)
        self.assertEqual(len(self.game.community_cards), 5)

    def test_NextStage_Flop_BotsSeeFiveCards(self):
        self.game.next_stage("preflop")
        self.game.next_stage("flop")
        for player in self.players:
            self.assertEqual(len(player.get_hand()), 5)

    def test_NextStage_River_BotsSeeSevenCards(self):
        for stage in ("preflop", "flop", "turn", "river"):
            self.game.next_stage(stage)
        for player in self.players:
            self.assertEqual(len(player.get_hand()), 7)

    def test_FirstBotActs_RaisesAllInAndPotGrows(self):
        action = self.play_action(self.player_a)
        self.assertEqual(action, "raise")
        self.assertEqual(self.player_a.get_chips(), 0)
        self.assertEqual(self.game.get_pot(), 1000)
        self.assertEqual(self.game.get_table_bet(), 1000)

    def test_SecondBotFacesAllIn_SameStack_Calls(self):
        self.play_action(self.player_a)  # a shoves 1000
        action = self.play_action(self.player_b)  # b has exactly 1000, cannot raise over it
        self.assertEqual(action, "call")
        self.assertEqual(self.player_b.get_chips(), 0)
        self.assertEqual(self.game.get_pot(), 2000)

    def test_AllThreeAllIn_PotIsAllChips(self):
        for player in self.players:
            self.play_action(player)
        self.assertEqual(self.game.get_pot(), 3000)
        for player in self.players:
            self.assertEqual(player.get_chips(), 0)
            self.assertEqual(player.get_current_bet(), 1000)

    def test_ShortStackFacesBigShove_CallsForWhatItHas(self):
        self.player_b.lose_chips(700)  # b only has 300
        self.play_action(self.player_a)  # a shoves 1000
        action = self.play_action(self.player_b)
        self.assertEqual(action, "call")
        self.assertEqual(self.player_b.get_chips(), 0)
        self.assertEqual(self.player_b.get_current_bet(), 300)
        self.assertEqual(self.game.get_pot(), 1300)

    def test_BigStackFacesShortShove_Raises(self):
        self.player_a.lose_chips(800)  # a has 200
        self.play_action(self.player_a)  # a shoves 200
        action = self.play_action(self.player_b)  # b has 1000, can raise over 200
        self.assertEqual(action, "raise")
        self.assertEqual(self.game.get_table_bet(), 1000)
        self.assertEqual(self.game.get_pot(), 1200)

    def test_FoldedGameBot_HandIsCleared(self):
        self.game.start_hand()
        self.player_c.fold()
        self.assertEqual(self.player_c.get_hand(), [])
        self.assertTrue(self.player_c.is_out)

    def test_RoundReset_AfterAllIn_ClearsBetsKeepsPot(self):
        for player in self.players:
            self.play_action(player)
        for player in self.players:
            player.reset_round()
        self.game.table_bet = 0
        self.assertEqual(self.game.get_pot(), 3000)
        for player in self.players:
            self.assertEqual(player.get_current_bet(), 0)

    def test_FullHandAllIn_ShowdownRunsWithoutError(self):
        for stage in ("preflop", "flop", "turn", "river"):
            self.game.next_stage(stage)
        for player in self.players:
            self.play_action(player)
        self.game.next_stage("showdown")
        self.assertEqual(self.game.get_pot(), 3000)

    def test_DetermineHand_ForAllInBot_ReturnsKnownHandName(self):
        for stage in ("preflop", "flop", "turn", "river"):
            self.game.next_stage(stage)
        score, name = self.game.determine_hand(self.player_a.get_hand()[:2])
        self.assertIn(name, Game.HAND_NAMES)


unittest.main(argv=[""], exit=False, verbosity=2)

test_AllInBot_IsABaseBot (__main__.AllInBot_Core_UnitTests.test_AllInBot_IsABaseBot) ... ok
test_AlreadyAllIn_NoChipsLeft_Calls (__main__.AllInBot_Core_UnitTests.test_AlreadyAllIn_NoChipsLeft_Calls) ... ok
test_AlreadyBetSome_RaiseTargetIsBetPlusChips (__main__.AllInBot_Core_UnitTests.test_AlreadyBetSome_RaiseTargetIsBetPlusChips) ... ok
test_ChooseAction_DoesNotChangeChips (__main__.AllInBot_Core_UnitTests.test_ChooseAction_DoesNotChangeChips) ... ok
test_FacingSmallerBet_RaisesAllChips (__main__.AllInBot_Core_UnitTests.test_FacingSmallerBet_RaisesAllChips) ... ok
test_NoBetsYet_RaisesAllChips (__main__.AllInBot_Core_UnitTests.test_NoBetsYet_RaisesAllChips) ... ok
test_RaiseTarget_MatchesChipsAllInWouldPutIn (__main__.AllInBot_Core_UnitTests.test_RaiseTarget_MatchesChipsAllInWouldPutIn) ... ok
test_StackCannotCoverBet_Calls (__main__.AllInBot_Core_UnitTests.test_StackCannotCoverBet_Calls) ... ok
test_StackExactlyMatchesBet_Calls (__main__.AllInBot_Core_UnitTests.test_StackExactlyMatch

preflop
flop: revealing three cards
turn: revealing penultimate card
river: revealing final card
preflop
flop: revealing three cards
turn: revealing penultimate card
river: revealing final card
showdown
preflop
flop: revealing three cards
preflop
flop: revealing three cards
turn: revealing penultimate card
river: revealing final card
